# Building the Taxonomy

#### Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Go up one level from 'Notebooks' to the project root
project_root = os.path.abspath(os.path.join('..'))

# Add the project root to the system path
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from pyvent.tools.llm.openai_api import OpenAIAgent
import nest_asyncio
nest_asyncio.apply()
import Function_Files.Load_Isolate_functions as lif
import Function_Files.Classification_functions as cf
import datetime

#### Variables

In [14]:
CATEGORY = "Soap & Sanitizer"
today = datetime.datetime.now().strftime('%m.%d.%Y')

ip_path = "C:\\Users\\ZWayne\\OneDrive - Advent International\\Advent Labs - Documents\\02. Portfolio Companies\\Imperial Dade\\Category Mgmt Project\\Data\\Input\\"
OP_PATH = f"C:\\Users\\ZWayne\\OneDrive - Advent International\\Documents\\GitHub\\ImperialDadeCategoryManagement\\Outputs\\{CATEGORY}\\"

#### Read in data
sfy = salsify, to_keep = item_master mixed with Country of Origin file to get skus with >0 L3M Sales, Volume and GM

In [8]:
#Read in files, define variables.
sfy = pd.read_excel(f"{ip_path}All Salsify Items.xlsx", sheet_name='in', skiprows=1)
to_keep = pd.read_csv(f"{OP_PATH}keep_list_05.27.2025.csv")
#rebates = pd.read_excel(f"{ip_path}2024 Top 250 Incremental and % Gross 2024-12-12 With MBR.xlsx", sheet_name='% of Gross', header = 2)

C:\Users\zwayne\AppData\Local\Temp\ipykernel_21348\3606080615.py:3: DtypeWarning: Columns (4,28,31,37,41,42,49,51,53,54,58,61,65,72,74,75,76,77,78,82) have mixed types. Specify dtype option on import or set low_memory=False.
  to_keep = pd.read_csv(f"{OP_PATH}keep_list_05.27.2025.csv")


In [15]:
# Create Entity--Item column for cluster and sfy dataframes
sfy['Entity--Item'] = '1--'+sfy['S2K Item Number'].astype(str).str.strip().str.upper()
im_grp = to_keep.copy()

#### Get S2k Items

In [16]:
# Identify all items in category both A) using cleaned s2k cat
# get_columns_with_coverage is a function that takes in a category and returns the columns with coverage for that category.
S2K_DIV = 'Soap & Sanitizer - 528' 
im_s2k, columns_with_coverage, example_data = lif.get_columns_with_coverage(S2K_DIV, im_grp, sfy, 15)

Processing category 'Soap & Sanitizer - 528' at level 2
Filtered sfy to 905 rows
Found 14 columns meeting 15% coverage
Extracted sample data for 14 columns


In [17]:
example_data

,Hand Soap Product Type,Product Type Collapse,Skin & Personal Care Product Features,Pack Size,Color,Material,Scent,Chemical Product Form,Branded Product Name,Package Volume/Weight Value,Package Volume/Weight Preferred Metric,Package Volume/Weight,Package Volume (FLOZ),Product Attributes
0,Hand Soap,Laundry Detergent,70% Ethyl Alcohol|Advanced|Healthcare|Hypoalle...,4/Pail,Yellow,Chemicals,Fresh,Tablet,Powerball,1.20,Liter (L),1.2 L,40.576800,ChemicalsFreshTabletPowerball
1,Hand Soap,Dishmachine Detergent,80% Ethyl Alcohol|Cartridge|Refill,376/Case,Yellow,Chemicals,Cranberry,Gel,ES8,1.00,Liter (L),1 L,33.814000,ChemicalsGelES81.2Liter (L)1.2 L40.5768
2,Hand Soap,Hand Sanitizer,70% Ethyl Alcohol|Advanced,2/Case,White,Chemicals,Citrus Ginger,Solid,Solid Metal Pro,2.00,Fluid Ounces (FLOZ),2 FLOZ,2.000000,Chemicals
3,Hand Soap,Laundry Detergent,Advanced Moisturizers|Refill,6/Case,Clear,Chemicals,Pleasant,Foam,All Ultra,1.20,Liter (L),1.2 L,40.576800,ChemicalsSolidSolid Metal Pro
4,Hand Soap,Warewashing Detergent,ADX-12,4/Case,Yellow,Chemicals,Plum,Powder,TFX,1200.00,Milliliters (mL),1200 mL,40.576182,ChemicalsFoam1Liter (L)1 L33.814
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Handwash,Laundry Detergent,Antimicrobial|Counter Mount|Refill|Fragrance F...,94/Box,Clear,Chemicals,Unscented,Liquid,RD - Pink Hand Soap,1.00,Fluid Ounces (FLOZ),1 FLOZ,42.267500,ClearChemicalsUnscentedFoam1.2Liter (L)1.2 L40...
96,Hand Soap,Hand Soap,Antibacterial|Moisturizing|Refill,6/Case,Black,Chemicals,Fresh,Foam,SUPRO MAX,1.00,Fluid Ounces (FLOZ),1 FLOZ,23.669439,ClearChemicalsUnscentedFoamLTX-7700Milliliters...
97,Hand Soap,Hand Soap,Mild|Fragrance Free|Dye Free|Hypoallergenic|Re...,4/Case,Blue|White,Chemicals,Almond,Foam,FMX-12 CHG,1.25,Liter (L),1.25 L,128.000000,ChemicalsUnscentedFoamYes1.6Liter (L)1.6 L54.1024
98,Hand Soap,Manual Dish Detergent,Alcohol Free,4/Case,Clear,Chemicals,Characteristic,Foam,LTX-12,1.20,Liter (L),1.2 L,77.772200,ChemicalsLiquid1Gallon (GAL)1 GAL128


In [18]:
columns_with_coverage

['Hand Soap Product Type',
 'Product Type Collapse',
 'Skin & Personal Care Product Features',
 'Pack Size',
 'Color',
 'Material',
 'Scent',
 'Chemical Product Form',
 'Branded Product Name',
 'Package Volume/Weight Value',
 'Package Volume/Weight Preferred Metric',
 'Package Volume/Weight',
 'Package Volume (FLOZ)',
 'Product Attributes']

In [19]:
# Can manually check/change the columns with coverage to see if they are correct
columns_for_description = ['Hand Soap Product Type',
 'Skin & Personal Care Product Features',
 'Pack Size',
 'Color',
 'Material',
 'Scent',
 'Chemical Product Form',
 'Branded Product Name',
 'Package Volume/Weight',
 'Package Volume (FLOZ)',
 'Product Attributes']

#### Get Non-S2K items

In [20]:
im_s2k

,Entity--Item,L3M_Sales,L3M_Cogs,L3M_adj_vol,Top_Country_By_COGS,COGS_From_Top_Country,%_COGS_From_Top_Country,Vendor 1,Vendor 1 COGS,Vendor 1 % COGS,...,manufacturer_name,manufacturer_item_code,hazard_code,average_cost_amt,item_category_level1_code,item_category_level1_name,branch_item_category_key_concat,l4m_usage_qty,uom_group_type,victoriabay_code
42,1--.DIA82838,1937.00,1108.01,23.0,USA,1108.01,1.0,ESSENDANT RECEIVABLES LLC,1108.01,1.0,...,NaN,NaN,NaN,41.2900,NaN,NaN,1--528-1,0.000000,CS,N
47,1--.E19-2,787.70,542.64,10.0,USA,542.64,1.0,ECOLOGIC SOLUTIONS,542.64,1.0,...,NaN,NaN,NaN,0.0000,NaN,NaN,1--528-5,0.000000,CS,N
55,1--.N2-G,872.30,646.24,7.0,USA,646.24,1.0,ECOLOGIC SOLUTIONS,646.24,1.0,...,NaN,NaN,NaN,46.3500,NaN,NaN,1--528-8,0.000000,CS,N
221,1--057359,8640.02,7794.39,151.0,USA,7794.39,1.0,DIVERSEY,7794.39,1.0,...,NaN,NaN,NaN,90.4100,NaN,NaN,1--528-10,35.421687,CS,N
276,1--09104,1547.79,1166.00,11.0,USA,1166.00,1.0,SC JOHNSON PROFESSIONAL / DEB,1166.00,1.0,...,NaN,NaN,NaN,0.0000,NaN,NaN,1--528-1,0.000000,CS,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32520,1--VIP15,2054.59,1364.82,6.0,USA,1364.82,1.0,CLEANSLATE GROUP LLC,1364.82,1.0,...,NaN,NaN,NaN,153.1999,NaN,NaN,1--528-8,0.000000,DR,N
32521,1--VIP5,14053.92,10100.76,123.0,USA,10100.76,1.0,CLEANSLATE GROUP LLC,10100.76,1.0,...,NaN,NaN,NaN,52.2494,NaN,NaN,1--528-8,0.000000,PL,N
32892,1--WHITEPEARL,10360.30,6600.45,239.0,USA,6600.45,1.0,NATIONAL CHEMICAL LABORATORIES,6602.12,1.0,...,NaN,NaN,NaN,28.1200,NaN,NaN,1--528-1,0.000000,CS,N
33500,1--Z650VILVER,2058.00,1819.30,14.0,USA,1819.30,1.0,ZOGICS LLC,1819.30,1.0,...,NaN,NaN,NaN,129.9500,NaN,NaN,1--528-7,0.000000,EA,N


In [21]:
# get_non_s2k is a function that takes in a strings to search, filters the im_grp dataframe for those strings, runs those rows through an LLM model, and returns the results.
search_str = ['soap', 'sanitiz','detergent','clean', 'purell','sani','clnr','deter','disinfect', 'foam','rinse','soap']
im_nons2k = lif.get_non_s2k(im_grp, search_str, category_name = CATEGORY, extra = "This includes dish soap and detergent.")


Processing non-primary items for category: Soap & Sanitizer
Found 5338 non-primary items matching search pattern
Consider submitting unique prompts to the API to save on costs and time


Running cost $0.0000:  29%|██▉       | 49/167 [02:07<05:07,  2.60s/chunk]


KeyboardInterrupt: 

#### Concatenate S2K and Non-S2K

In [ ]:
# combine im_s2k and im_nons2k dataframes
im_concat = pd.concat([im_s2k, im_nons2k], axis=0); print(im_concat.shape)

(3766, 88)


In [ ]:
# get columns with coverage from sfy dataframe and merge with im_concat dataframe
sfy_covered = sfy[columns_with_coverage+['Entity--Item']].copy()
im_final = im_concat.merge(sfy_covered, on='Entity--Item', how='left', suffixes=('', '_dup'))

In [ ]:
im_final[im_final.duplicated(subset=['Entity--Item'], keep=False)]
# keep first and set columns_with_coverage to nan
im_final = im_final.drop_duplicates(subset=['Entity--Item'], keep='first')
for col in columns_with_coverage:
    if col in im_final.columns:
        im_final[col] = im_final[col].fillna('')

#### Write files - this is the subsection of to_keep that fits the category

In [ ]:
os.makedirs(OP_PATH, exist_ok=True)  
im_final.to_csv(f"{OP_PATH}{CATEGORY}_SKUS_{today}.csv", index=False)

In [21]:
im_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_SKUS_{today}.csv")

## Classification

#### Generate parts for taxonomy prompt

In [22]:
# generate a string of the top 5 values for each column in columns_for_description to help with the prompt
prompt_options_string = cf.get_top_values(im_final, columns_for_description, 5)
prompt_options_string

'Hand Soap Product Type: Hand Soap, Hand Cleaner, Handwash \n\n Skin & Personal Care Product Features: Refill, Antibacterial|Refill, Antibacterial, Touchless, Push Style \n\n Pack Size: 4/Case, 2/Case, 6/Case, 1/Pail, 1/Each \n\n Color: Clear, White, Blue, Pink, Yellow \n\n Material: Chemicals, Plastic, Wood, Chlorine, Chlorite \n\n Scent: Unscented, Fresh, Lemon, Floral, Cranberry \n\n Chemical Product Form: Foam, Liquid, Gel, Powder, Solid \n\n Branded Product Name: TFX, Professional, ADX-12, CXM/CXI/CXT, LTX-12 \n\n Package Volume/Weight: 1 GAL, 1.2 L, 1 L, 1.25 L, 5 GAL \n\n Package Volume (FLOZ): 128.0, 40.5768, 33.814, 42.2675, 640.0 \n\n Product Attributes: Chemicals, ChemicalsLiquid1Gallon (GAL)1 GAL128, ClearChemicalsUnscentedFoam1.2Liter (L)1.2 L40.5768, ChemicalsLiquid5Gallon (GAL)5 GAL640, ClearChemicalsUnscentedFoam1Liter (L)1 L33.814'

In [23]:
# create a description string for each row in that will be used in the promp
im_final['All Descriptions'] = (
    im_final['description_line1_txt'].fillna('') + ' ' +
    im_final['description_line2_txt'].fillna('') + ' ' +
    im_final['description_line3_txt'].fillna('')
    ).str.strip()

existing_columns_for_description = [col for col in columns_for_description if col in im_final.columns]

if not existing_columns_for_description:
    print("Warning: None of the specified columns for description exist in the DataFrame.")
    im_final['description'] = "" # Create an empty description column
else:
    if len(existing_columns_for_description) < len(columns_for_description):
        missing_cols = set(columns_for_description) - set(existing_columns_for_description)
        print(f"Warning: The following specified columns were not found in the DataFrame and will be skipped: {missing_cols}")
    
im_final['description'] = im_final.apply(
    lambda row: cf.create_description_string(row, existing_columns_for_description),
    axis=1 # Apply function row-wise
)

In [24]:
output_str = cf.get_most_common_values(prompt_options_string)

In [25]:
output_str

'Hand Soap Product Type: Hand Soap | Skin & Personal Care Product Features: Refill | Pack Size: 4/Case | Color: Clear | Material: Chemicals | Scent: Unscented | Chemical Product Form: Foam | Branded Product Name: TFX | Package Volume/Weight: 1 GAL | Package Volume (FLOZ): 128.0 | Product Attributes: Chemicals'

In [26]:
example_desc, example_output = cf.explain_top_sales_description(im_final, output_str)

Running cost $0.0000: 100%|██████████| 1/1 [00:01<00:00,  1.32s/chunk]


#### Taxonomy prompts

In [27]:
model = 'gpt-4o-mini' 
chunk_size = 32       
agent = OpenAIAgent(model=model, chunk_size=chunk_size)

In [28]:
system_prompt = f"""

You are to pull out information from a product description. 

Here are the attributes I am looking for:
{columns_for_description}

Here is a look at the most common values for each of these attributes. This is not an exhaustive list, but it should help you understand what I am looking for. 
You probably will need to pull out similar information for an attribute that's not on this list, except for Subcategory - for Subcategory only choose one of the given options. Pick the one that fits best for Subcategory - put Other if none are perfect: \n\n
{prompt_options_string}

Return in 'Column: Value' format separated by '|'. For example, if the description is "{example_desc}", then return
{example_output}
"""
user_prompt = "Pull out the relevant information based on this product description '{description}'"

im_final = agent.format_df_prompts(im_final, system_prompt, user_prompt)
im_final = agent.run_df_prompts(im_final)

Consider submitting unique prompts to the API to save on costs and time


Running cost $0.6186: 100%|██████████| 118/118 [03:11<00:00,  1.62s/chunk]


#### Write file with just id and taxonomy

In [32]:
im_final_attributed, extract_attributes_to_dataframe = cf.extract_attributes_to_dataframe(im_final, columns_for_description, output_excel_filepath=f"{OP_PATH}{CATEGORY}_taxonomy_{today}.xlsx", vendor_col = 'vgn_name')

Processing 3766 rows from the input DataFrame...
Successfully created DataFrame with 3766 rows and 14 columns.
Successfully wrote DataFrame to Excel: C:\Users\ZWayne\OneDrive - Advent International\Advent Labs - Documents\02. Portfolio Companies\Imperial Dade\Category Mgmt Project\Data\Output\Soap & Sanitizer_taxonomy_08.15.2025.xlsx


In [33]:
cf.coverage_improvement(sfy, im_final, extract_attributes_to_dataframe, columns_for_description)

Computing coverage improvement analysis...
Computed post-LLM coverage for 14 columns
Computed initial coverage for 12 columns
Coverage improvement analysis complete. Found 11 columns to compare.


,% Coverage Post LLM,% Initial Coverage,Difference
Column_Name,,,
Hand Soap Product Type,82.90%,9.35%,73.55%
Skin & Personal Care Product Features,18.99%,10.09%,8.90%
Pack Size,93.76%,24.03%,69.73%
Color,17.05%,7.41%,9.64%
Material,51.49%,20.31%,31.17%
Scent,23.55%,7.86%,15.69%
Chemical Product Form,82.08%,15.40%,66.68%
Branded Product Name,91.98%,4.83%,87.15%
Package Volume/Weight,86.64%,15.03%,71.61%


#### write file with attributes

In [34]:
im_final_attributed.to_csv(f"{OP_PATH}{CATEGORY}_Attributed_{today}.csv", index=False)